# Yandu Wang — WIP notebook

# update 9.1

## 0. Paths

Works on Colab and locally. It looks for a known file instead of assuming a folder layout.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

GROUP_ID = "Group001"

try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    pass

_colab = Path("/content/drive/MyDrive/Group001_A1")
BASE = _colab if _colab.is_dir() else Path(".")


def find_dir(marker, candidates):
    """Return the first candidate folder holding `marker`, or raise.

    No recursive search. A recursive fallback would quietly pick up the
    DEC-022 placeholder tables in outputs_wip_jasmine/provisional/, which have
    the same shape as the real ones — we would validate half-finished tables
    and every check would still pass. Failing here is the point.
    """
    for folder in candidates:
        if (folder / marker).exists():
            return folder
    tried = "\n  ".join(str(c) for c in candidates)
    raise FileNotFoundError(
        f"{marker} not found. Looked in:\n  {tried}\n"
        f"If outputs_wip_jasmine/provisional/ exists, the pipeline ran without "
        f"WP4's text functions and those tables are placeholders (DEC-022)."
    )


INPUT_DIR = find_dir(f"{GROUP_ID}_commerce.json", [
    BASE / "DATA" / f"{GROUP_ID}_A1" / "raw_input",
    BASE / "raw_input",
])

TABLE_DIR = find_dir(f"{GROUP_ID}_orders_standardised.csv", [
    BASE / "01_WIP" / "outputs_wip_jasmine",
    BASE / "02_Outputs",
    BASE,
])

DICT_PATH = find_dir("public_data_dictionary.csv", [
    BASE / "DATA" / f"{GROUP_ID}_A1",
    BASE,
]) / "public_data_dictionary.csv"

# DEC-022 guard: refuse to validate placeholder tables even if they resolve.
if "provisional" in TABLE_DIR.parts:
    raise RuntimeError(
        f"TABLE_DIR resolved to {TABLE_DIR}, which is the DEC-022 placeholder "
        f"output. Those tables were built without WP4's text functions and are "
        f"indistinguishable from the real ones by shape alone. Validate the real "
        f"export instead."
    )

# WP3's own outputs live here, never in the shared data folder.
OUTPUT_DIR = BASE / "outputs_wip_yandu"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

JSON_PATH = INPUT_DIR / f"{GROUP_ID}_commerce.json"
XML_PATH  = INPUT_DIR / f"{GROUP_ID}_operations.xml"

print("raw input :", INPUT_DIR)
print("tables    :", TABLE_DIR)
print("dictionary:", DICT_PATH)
print("my output :", OUTPUT_DIR)

raw input : DATA/Group001_A1/raw_input
tables    : .
dictionary: DATA/Group001_A1/public_data_dictionary.csv
my output : outputs_wip_yandu


### 0.1 Load the six tables and the data dictionary

Part B checks the six tables Jasmine exported. Two read options matter:

- `keep_default_na=False` — pandas would otherwise turn the text `NaN` into an empty value, and
  that text is exactly what we need to see (§6.7).
- `dtype=str` — keeps leading zeros on IDs and postcodes.

In [2]:
TABLES = ["orders", "order_items", "customers", "deliveries", "products", "product_reviews"]

T = {name: pd.read_csv(TABLE_DIR / f"{GROUP_ID}_{name}_standardised.csv",
                       keep_default_na=False, dtype=str)
     for name in TABLES}

dd = pd.read_csv(DICT_PATH)      # the data dictionary: what each table should look like

for name, df in T.items():
    print(f"{name:16s} {len(df):>7,} rows x {df.shape[1]:>2} cols")

orders             5,000 rows x 23 cols
order_items       15,685 rows x  6 cols
customers            500 rows x 20 cols
deliveries         5,000 rows x 20 cols
products           1,000 rows x 21 cols
product_reviews    7,000 rows x 21 cols


### 0.2 How the checks compare values

Three helpers every check uses. Both rules come from WP2 §4.0.3 — using the obvious version
instead reports 40 failures on `order_total` that are not real problems.

- **Rounding money.** Python's `round()`, not pandas `.round(2)`. The two disagree when a value
  sits exactly halfway, and the source file was made with Python's version.
- **Comparing numbers.** The tolerance is 0.01 and it means *at most* 0.01, not *less than*.
  `equal_nan=True` is needed because a missing value is not equal to itself in pandas, so two
  missing values would look like a conflict.

In [3]:
TOLERANCE = 0.01


def money_round(series):
    """Round money the same way the source file did: Python's round(), not pandas .round(2)."""
    return series.map(lambda v: round(v, 2))


def same_number(a, b, atol=TOLERANCE):
    """True when a and b are within 0.01. Two missing values count as the same."""
    return np.isclose(a, b, rtol=0, atol=atol, equal_nan=True)


RESULTS = []


def record(val_id, passed, evidence, note=""):
    """Write down one check: its ID, its status, what we saw, and what it means.

    `passed` takes three values, not two:
        True   -> PASS
        False  -> FAIL
        None   -> NOT RUN, for a check that is specified but cannot run here.

    NOT RUN is a status, not a silent omission. A register that only lists the
    checks it managed to run is not an honest account of the ones it specified.
    """
    status = "NOT RUN" if passed is None else ("PASS" if passed else "FAIL")
    RESULTS.append({"id": val_id, "status": status,
                    "evidence": evidence, "note": note})
    print(f"{val_id:18s} {status:8s} {evidence}")

---

# Part A — G0 count (23 Aug)

### A.1 Read the two files

ues json and xml.etree.ElementTree

In [4]:
import json
import xml.etree.ElementTree as ET

# ---- JSON (commerce out) ----
with open(JSON_PATH, encoding="utf-8") as f:
    jdata = json.load(f)

print("JSON top:", list(jdata.keys()))
print("orders list lenth:", len(jdata["orders"]))

# ---- XML (operations ERP out) ----
xroot = ET.parse(XML_PATH).getroot()
n_order_nodes = len(list(xroot.iter("Order")))
print("XML <Order> point number:", n_order_nodes)


JSON top: ['customerProfiles', 'exportMetadata', 'orders', 'productReviews']
orders list lenth: 2818
XML <Order> point number: 2818


### A.2 Get order_id
# Count orders from Header/Order_ID only — it also repeats in items and delivery.

In [5]:
# Values ​​(retaining duplicates)
json_ids = [o["header"]["orderID"] for o in jdata["orders"]]
xml_ids = []
for order in xroot.iter("Order"):
    header = order.find("Header")
    if header is not None:
        oid = header.find("Order_ID")
        if oid is not None and oid.text:
            xml_ids.append(oid.text.strip())

print("JSON Number of order records (Contains duplicates):", len(json_ids))
print("XML  Number of order records (Contains duplicates):", len(xml_ids))


JSON Number of order records (Contains duplicates): 2818
XML  Number of order records (Contains duplicates): 2818


### A.3 Within-file duplicate count

In [6]:
from collections import Counter

jc = Counter(json_ids)
xc = Counter(xml_ids)

json_within_dups = sum(1 for _id, n in jc.items() if n > 1)
xml_within_dups  = sum(1 for _id, n in xc.items() if n > 1)

print("JSON Number of duplicate order_ids in the file:", json_within_dups)
print("XML  Number of duplicate order_ids in the file:", xml_within_dups)

WITHIN_FILE_DUPLICATE_COUNT = json_within_dups
print("within-file duplicate count =", WITHIN_FILE_DUPLICATE_COUNT)


JSON Number of duplicate order_ids in the file: 68
XML  Number of duplicate order_ids in the file: 68
within-file duplicate count = 68


### A.4 Cross-file overlap count

In [7]:
json_unique = set(json_ids)
xml_unique  = set(xml_ids)

overlap_ids = json_unique & xml_unique
CROSS_FILE_OVERLAP_COUNT = len(overlap_ids)

print("JSON Different after deduplication order_id:", len(json_unique))
print("XML  Different after deduplication order_id:", len(xml_unique))
print("cross-file overlap count =", CROSS_FILE_OVERLAP_COUNT)


JSON Different after deduplication order_id: 2750
XML  Different after deduplication order_id: 2750
cross-file overlap count = 500


### A.5 Canonical order count

Remove duplicates, then merge the two files; what remains is the actual number of orders.

In [8]:
canonical_ids = json_unique | xml_unique
CANONICAL_ORDER_COUNT = len(canonical_ids)

# all = J + X - dp
check = len(json_unique) + len(xml_unique) - CROSS_FILE_OVERLAP_COUNT
assert check == CANONICAL_ORDER_COUNT, (check, CANONICAL_ORDER_COUNT)

print(">>> canonical order count =", CANONICAL_ORDER_COUNT)


>>> canonical order count = 5000


### A.6 The three numbers

In [9]:

print(f"  1. canonical order count       = {CANONICAL_ORDER_COUNT}")
print(f"  2. within-file duplicate count = {WITHIN_FILE_DUPLICATE_COUNT}  (each file: {WITHIN_FILE_DUPLICATE_COUNT})")
print(f"  3. cross-file overlap count    = {CROSS_FILE_OVERLAP_COUNT}")
print(f"{len(json_unique)} + {len(xml_unique)} - {CROSS_FILE_OVERLAP_COUNT} = {CANONICAL_ORDER_COUNT}")

  1. canonical order count       = 5000
  2. within-file duplicate count = 68  (each file: 68)
  3. cross-file overlap count    = 500
2750 + 2750 - 500 = 5000


---

# Part B — the validation register

These cells carry the template's §6 numbering exactly (D4), so they move into
`Group001_solution.ipynb` at assembly without changing the mapping's
`notebook_evidence` column.

## 6. Validation register

Keep each check executable and give it a stable `VAL-...` ID. Immediately after
each code check, record the observed result, `PASS`/`FAIL`, evidence and
resolution/interpretation. A genuine, explained failure is preferable to a
fabricated pass.

Required areas include schema/types, primary and foreign keys, row flow and
source coverage, overlap, arithmetic, temporal logic, text/reference behaviour
and multilingual handling.

**Three rules that apply to every check below.**

1. **Compare after normalising, never before.** The XML writes money as `AUD 155.15`, booleans as
   `Y`, and dates as day-first; the JSON uses native types. Comparing before normalising reports
   format differences as conflicts that do not exist.
2. **Work out the expected number in the same cell that checks it.** Nothing is typed in by hand.
   Writing `assert len(orders) == 5000` loses the mark; working out the same 5,000 from the two
   files and then checking it keeps it.
3. **Say which source wins, do not leave it unsaid.** Deduplication puts the JSON rows first and
   keeps the first copy, so JSON wins. The values agree either way, so the choice changes nothing
   — but it is still a choice, and C2 marks whether we wrote it down.

**Primary keys** (DEC-016): `orders.order_id` · `order_items.order_item_id` ·
`customers.customer_id` · `deliveries.delivery_id` · `products.product_id` ·
`product_reviews.review_id`.

### 6.1 Schema and type checks (`VAL-SCHEMA-...`)

Does each table have the right columns, in the right order, with the right type? The data
dictionary says what each table should look like. If this is wrong, nothing after it means
anything, so it runs first.

| ID | What it checks | Table | Passes when | If it fails |
|---|---|---|---|---|
| VAL-SCHEMA-01 | Columns match the dictionary: same names, same order, same types | orders | exact match | Wrong columns break B1's field-order mark and every later join |
| VAL-SCHEMA-02 | Same check | order_items | exact match | as above |
| VAL-SCHEMA-03 | Same check | customers | exact match | as above |
| VAL-SCHEMA-04 | Same check | deliveries | exact match | as above |
| VAL-SCHEMA-05 | Same check | products | exact match | as above |
| VAL-SCHEMA-06 | Same check | product_reviews | exact match | as above |
| VAL-SCHEMA-07 | IDs and `home_postcode` are still text, with leading zeros kept | all six | all text | `home_postcode` is all digits, so pandas turns it into a number without warning and the zero is lost |
| VAL-SCHEMA-08 | No helper column reached the file — no `source_system`, no `_raw` | all six | none found | An internal column shipped in the submitted file |
| VAL-SCHEMA-09 | Dates are written the published way: `YYYY-MM-DD` for dates, `YYYY-MM-DD HH:MM:SS` for timestamps | all dated tables | exact match | A date field with a time on the end breaks the dictionary |
| VAL-SCHEMA-10 | Columns marked `nullable = False` have no empty cells | all six | no empty cells | A required field is blank |
| VAL-SCHEMA-11 | The six files have the required names, including `_standardised` | output folder | all six present | B1's top band starts with "present with the required filenames" |

**A note on `nullable = False`.** All 21 `product_reviews` columns are marked `nullable = False`,
but the spec also says missing text must be written as `NaN`. Both cannot be true at once, so
VAL-SCHEMA-10 reads `nullable = False` as *no empty cell*, not *no `NaN`*. Every table passes it
today. Worth a `DEC-` row either way, because the check is written differently under each
reading.

In [10]:
# --- §6.1 checks ---
# The dictionary says what each table should look like. We compare against it,
# never against a list typed in by hand.

for name, df in T.items():
    want = dd[dd.output_table == name].sort_values("position")["field_name"].tolist()
    got  = list(df.columns)
    record(f"VAL-SCHEMA-{name}", got == want,
           f"{len(got)} columns, order matches the dictionary" if got == want
           else f"got {got[:3]}..., wanted {want[:3]}...")

# VAL-SCHEMA-07 — IDs and postcode must stay text with leading zeros.
# home_postcode is the risky one: it is all digits, so pandas converts it silently.
pc = T["customers"]["home_postcode"]
record("VAL-SCHEMA-07", pc.str.fullmatch(r"\d{4}").all(),
       f"home_postcode is 4-character text on every row ({pc.min()}..{pc.max()})")

# VAL-SCHEMA-08 — no helper column should reach the exported file.
leaked = [f"{n}.{c}" for n, df in T.items() for c in df.columns
          if c == "source_system" or c.endswith("_raw")]
record("VAL-SCHEMA-08", not leaked, f"helper columns found: {leaked or 'none'}")

# VAL-SCHEMA-09 — dates written the published way.
DATE   = r"\d{4}-\d{2}-\d{2}"
STAMP  = r"\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}"
date_fields  = [("customers","signup_date"), ("deliveries","dispatch_date"),
                ("deliveries","promised_date"), ("deliveries","delivered_date"),
                ("products","launch_date")]
stamp_fields = [("orders","order_timestamp"), ("product_reviews","review_timestamp")]
bad = [f"{n}.{c}" for n, c in date_fields  if not T[n][c].str.fullmatch(DATE).all()]
bad += [f"{n}.{c}" for n, c in stamp_fields if not T[n][c].str.fullmatch(STAMP).all()]
record("VAL-SCHEMA-09", not bad, f"wrongly formatted date fields: {bad or 'none'}")

# VAL-SCHEMA-10 — nullable=False means no empty cell (see the note above).
empties = {f"{n}.{c}": int((T[n][c] == "").sum())
           for n in T
           for c in dd[(dd.output_table == n) & (dd.nullable == False)].field_name
           if (T[n][c] == "").any()}
record("VAL-SCHEMA-10", not empties, f"empty cells in nullable=False fields: {empties or 'none'}")

# VAL-SCHEMA-11 — the six files carry the required names.
wanted = {f"{GROUP_ID}_{n}_standardised.csv" for n in TABLES}
found  = {p.name for p in TABLE_DIR.glob("*_standardised.csv")}
record("VAL-SCHEMA-11", wanted <= found,
       f"{len(wanted & found)} of 6 required filenames present; missing {sorted(wanted - found) or 'none'}")


VAL-SCHEMA-orders  PASS     23 columns, order matches the dictionary
VAL-SCHEMA-order_items PASS     6 columns, order matches the dictionary
VAL-SCHEMA-customers PASS     20 columns, order matches the dictionary
VAL-SCHEMA-deliveries PASS     20 columns, order matches the dictionary
VAL-SCHEMA-products PASS     21 columns, order matches the dictionary
VAL-SCHEMA-product_reviews PASS     21 columns, order matches the dictionary
VAL-SCHEMA-07      PASS     home_postcode is 4-character text on every row (3000..3182)
VAL-SCHEMA-08      PASS     helper columns found: none
VAL-SCHEMA-09      PASS     wrongly formatted date fields: none
VAL-SCHEMA-10      PASS     empty cells in nullable=False fields: none
VAL-SCHEMA-11      PASS     6 of 6 required filenames present; missing none


**Observed result / status / interpretation.** All six field sets match the dictionary in
name, order and count. `home_postcode` survives as four-character text, so the one column that
pandas would convert without warning did not. No helper column reached any file, which is the
evidence that `conform_to_contract` still strips the overlap marker WP2 adds before export. No
`nullable = False` field holds an empty cell anywhere — that is what makes the reading proposed
above checkable rather than rhetorical, since every absent value is the literal sentinel instead.

**What would have made this section fail.** A renamed or reordered column, a postcode read as a
number, `source_system` surviving into a CSV, or a date written with a time appended. Echo
planted a blank `order_id` in a copy of the data and VAL-SCHEMA-10 caught it, so the section is
sensitive rather than merely quiet.

### 6.2 Primary- and foreign-key checks (`VAL-PK-...`, `VAL-FK-...`)

A **primary key** is the column that names each row. It has to be unique and never empty.
A **foreign key** is a column pointing at another table's primary key — it has to point at a row
that exists.

| ID | What it checks | Table | Passes when | If it fails |
|---|---|---|---|---|
| VAL-PK-01 | `order_id` is unique and never empty | orders | unique, no blanks | Two rows share a name, so every foreign key pointing here is unreliable |
| VAL-PK-02 | `order_item_id` is unique and never empty | order_items | unique, no blanks | as above |
| VAL-PK-03 | `customer_id` is unique and never empty | customers | unique, no blanks | as above |
| VAL-PK-04 | `delivery_id` is unique and never empty | deliveries | unique, no blanks | as above |
| VAL-PK-05 | `product_id` is unique and never empty | products | unique, no blanks | as above |
| VAL-PK-06 | `review_id` is unique and never empty | product_reviews | unique, no blanks | as above |

The eight foreign keys the spec asks for. These are checked on the **exported files**, not on the
sources — §1.3d already showed the sources are sound, but that is a different claim from the one
B1 and C2 mark.

| ID | Points from | Points to | Passes when | If it fails |
|---|---|---|---|---|
| VAL-FK-01 | `orders.customer_id` | `customers.customer_id` | nothing unmatched | An order belongs to a customer who does not exist |
| VAL-FK-02 | `order_items.order_id` | `orders.order_id` | nothing unmatched | An item belongs to no order, so revenue totals are wrong |
| VAL-FK-03 | `order_items.product_id` | `products.product_id` | nothing unmatched | An item points at a product not in the catalogue |
| VAL-FK-04 | `deliveries.order_id` | `orders.order_id` | nothing unmatched | A delivery exists for no order |
| VAL-FK-05 | `product_reviews.order_id` | `orders.order_id` | nothing unmatched | A review belongs to no order |
| VAL-FK-06 | `product_reviews.order_item_id` | `order_items.order_item_id` | nothing unmatched | A review belongs to no item |
| VAL-FK-07 | `product_reviews.product_id` | `products.product_id` | nothing unmatched | A review points at a missing product |
| VAL-FK-08 | `product_reviews.customer_id` | `customers.customer_id` | nothing unmatched | A review belongs to an unknown customer |

| ID | What it checks | Tables | Passes when | If it fails |
|---|---|---|---|---|
| VAL-FK-09 | `deliveries` has one row per order — same row count, `order_id` unique in both | deliveries, orders | 1:1 | The dictionary says one row per completed order delivery; if it is not 1:1, any join here multiplies rows |

In [11]:
# --- §6.2 checks ---
PK = {"orders": "order_id", "order_items": "order_item_id", "customers": "customer_id",
      "deliveries": "delivery_id", "products": "product_id", "product_reviews": "review_id"}

for name, key in PK.items():
    col = T[name][key]
    ok = col.is_unique and (col != "").all()
    record(f"VAL-PK-{name}", ok,
           f"{col.nunique():,} different values in {len(col):,} rows, {int((col=='').sum())} blank")

# The eight foreign keys the spec lists, checked on the exported files.
FK = [("VAL-FK-01", "orders", "customer_id", "customers", "customer_id"),
      ("VAL-FK-02", "order_items", "order_id", "orders", "order_id"),
      ("VAL-FK-03", "order_items", "product_id", "products", "product_id"),
      ("VAL-FK-04", "deliveries", "order_id", "orders", "order_id"),
      ("VAL-FK-05", "product_reviews", "order_id", "orders", "order_id"),
      ("VAL-FK-06", "product_reviews", "order_item_id", "order_items", "order_item_id"),
      ("VAL-FK-07", "product_reviews", "product_id", "products", "product_id"),
      ("VAL-FK-08", "product_reviews", "customer_id", "customers", "customer_id")]

for vid, child_t, child_c, parent_t, parent_c in FK:
    child, parent = set(T[child_t][child_c]), set(T[parent_t][parent_c])
    missing = child - parent
    record(vid, not missing,
           f"{child_t}.{child_c} -> {parent_t}.{parent_c}: "
           f"{len(missing)} unmatched of {len(child):,} different values")

# VAL-FK-09 — deliveries must be one row per order, or any join here multiplies rows.
d = T["deliveries"]
record("VAL-FK-09", len(d) == d.order_id.nunique() == len(T["orders"]),
       f"deliveries {len(d):,} rows, {d.order_id.nunique():,} different order_id, "
       f"orders {len(T['orders']):,} rows")


VAL-PK-orders      PASS     5,000 different values in 5,000 rows, 0 blank
VAL-PK-order_items PASS     15,685 different values in 15,685 rows, 0 blank
VAL-PK-customers   PASS     500 different values in 500 rows, 0 blank
VAL-PK-deliveries  PASS     5,000 different values in 5,000 rows, 0 blank
VAL-PK-products    PASS     1,000 different values in 1,000 rows, 0 blank
VAL-PK-product_reviews PASS     7,000 different values in 7,000 rows, 0 blank
VAL-FK-01          PASS     orders.customer_id -> customers.customer_id: 0 unmatched of 500 different values
VAL-FK-02          PASS     order_items.order_id -> orders.order_id: 0 unmatched of 5,000 different values
VAL-FK-03          PASS     order_items.product_id -> products.product_id: 0 unmatched of 1,000 different values
VAL-FK-04          PASS     deliveries.order_id -> orders.order_id: 0 unmatched of 5,000 different values
VAL-FK-05          PASS     product_reviews.order_id -> orders.order_id: 0 unmatched of 3,993 different values
VAL-FK-0

**Observed result / status / interpretation.** All six primary keys are unique and complete,
and all eight required foreign keys resolve with nothing unmatched **on the exported files**.
That last part matters: §1.3d showed the source union is referentially sound, which is a
different claim from the one B1 and C2 assess. `deliveries` is confirmed one row per order, so
no delivery-side join can multiply rows.

**What would have made this section fail.** A duplicate or blank key, an orphan line item, a
review pointing at an order that is not there, or `deliveries` holding more than one row per
order.

### 6.3 Source coverage and reconciliation checks (`VAL-FLOW-...`)

Did the right number of rows come through, and can each row be traced back to a source? This is
the reconciliation part of WP3.

| ID | What it checks | Table | Passes when | If it fails |
|---|---|---|---|---|
| VAL-FLOW-01 | Row count equals the number of different `order_id` in the two files put together, worked out in the same cell | orders | exact match | Deduplication dropped rows or kept too many |
| VAL-FLOW-02 | Same, on `order_item_id` | order_items | exact match | as above |
| VAL-FLOW-03 | Same, on `customer_id` (JSON is the only source) | customers | exact match | as above |
| VAL-FLOW-04 | Same, on `delivery_id` | deliveries | exact match | as above |
| VAL-FLOW-05 | Same, on `product_id` (XML is the only source) | products | exact match | as above |
| VAL-FLOW-06 | Same, on `review_id` | product_reviews | exact match | as above |
| VAL-FLOW-07 | The rows removed by deduplication match the count worked out from each raw file | all affected | exact match | Deduplication removed too much or too little |
| VAL-FLOW-08 | Every exported row can be traced back to at least one source row | all six | none untraced | A row exists that no source produced |
| VAL-FLOW-09 | For orders in **both** files, every shared column agrees **after normalising**. Any disagreement is written down with its key and column, not dropped | orders, order_items, deliveries, product_reviews | no conflicts | A real conflict exists, and §4 needs a rule about which source wins instead of a free choice |
| VAL-FLOW-10 | Repeated keys inside one file are two identical copies, so keeping the first cannot change a value | affected tables | identical copies | Keeping the first is no longer safe to justify (DEC-017) |
| VAL-FLOW-11 | The conflict detector is tested on a small made-up table with a conflict planted in it, then run on the real data | test fixture, then real | fires on the fixture, quiet on the real data | "We found no conflicts" and "our detector never works" look identical without this |
| VAL-FLOW-12 | The `both` marker survives deduplication: `source_system` is `JSON`, `XML` or `both` on `<table>_marked`, and the `both` set matches the overlap worked out again from the two key sets | shared tables, before export | marker matches | We lose track of which rows came from both files, right where §5 needs it. Note the marker must **not** reach the CSVs — VAL-SCHEMA-08 checks that |
| VAL-FLOW-13 | No column is entirely `NaN` | all six | none | A column that is all sentinel means a real source column was lost upstream. Nothing crashes and no row count changes, so only this check finds it |
| VAL-FLOW-14 | Items per order stay inside the limit worked out from the source, not a number typed in | order_items | within the limit | A limit written as 1–10 could never fire, because the 10 came from duplicates |
| VAL-FLOW-15 | Joining `order_items` to `orders` gives the item row count, and no order total is worked out on the joined table | orders, order_items | join size matches | An order total worked out after the join is multiplied by the number of items. WP2 avoids this by grouping first; F1 asks for the check by name, and avoiding is not the same as showing |

In [12]:
# --- §6.3 checks ---
# Every expected number below is worked out from the raw files in this cell.
# Nothing is typed in.

j_ord  = {o["header"]["orderID"] for o in jdata["orders"]}
x_ord  = {o.find("Header").find("Order_ID").text.strip() for o in xroot.iter("Order")}
j_item = {i["orderItemID"] for o in jdata["orders"] for i in o["shoppingCart"]}
x_item = {i.find("Order_Item_ID").text.strip() for i in xroot.iter("Item")}
j_cust = {c["customerID"] for c in jdata["customerProfiles"]}
j_del  = {o["delivery"]["deliveryID"] for o in jdata["orders"]}
x_del  = {e.find("Delivery_ID").text.strip() for e in xroot.iter("Delivery")}
x_prod = {p.find("Product_ID").text.strip() for p in xroot.iter("Product")}
j_rev  = {r["reviewID"] for r in jdata["productReviews"]}
x_rev  = {r.find("Review_ID").text.strip() for r in xroot.iter("Review")}

EXPECTED = {"orders": j_ord | x_ord, "order_items": j_item | x_item,
            "customers": j_cust, "deliveries": j_del | x_del,
            "products": x_prod, "product_reviews": j_rev | x_rev}

for i, (name, keys) in enumerate(EXPECTED.items(), start=1):
    record(f"VAL-FLOW-{i:02d}", len(T[name]) == len(keys),
           f"{name} has {len(T[name]):,} rows; the two files together hold {len(keys):,} different keys")

# VAL-FLOW-07 — how many rows deduplication removed, worked out per source.
raw_json_orders = [o["header"]["orderID"] for o in jdata["orders"]]
raw_xml_orders  = [o.find("Header").find("Order_ID").text.strip() for o in xroot.iter("Order")]
removed_json = len(raw_json_orders) - len(set(raw_json_orders))
removed_xml  = len(raw_xml_orders)  - len(set(raw_xml_orders))
record("VAL-FLOW-07", removed_json == removed_xml,
       f"deduplication removes {removed_json} rows from the JSON and {removed_xml} from the XML")

# VAL-FLOW-08 — every exported row traces back to a source row.
untraced = {n: len(set(T[n][PK[n]]) - keys) for n, keys in EXPECTED.items()}
record("VAL-FLOW-08", not any(untraced.values()),
       f"rows with no source: {untraced}")

# VAL-FLOW-13 — a column that is entirely NaN means a real source column was lost.
# Nothing crashes and no row count changes, so only this check finds it.
all_nan = [f"{n}.{c}" for n, df in T.items() for c in df.columns if (df[c] == "NaN").all()]
record("VAL-FLOW-13", not all_nan, f"columns that are entirely NaN: {all_nan or 'none'}")

# VAL-FLOW-14 — the limit comes from the source, not from a number we chose.
biggest_cart = max(len(o["shoppingCart"]) for o in jdata["orders"])
per_order = T["order_items"].groupby("order_id").size()
record("VAL-FLOW-14", per_order.max() == biggest_cart,
       f"items per order run {per_order.min()} to {per_order.max()}; "
       f"the biggest cart in the source holds {biggest_cart}")

# VAL-FLOW-15 — show the join multiplication instead of only avoiding it.
joined = T["order_items"].merge(T["orders"][["order_id"]], on="order_id")
record("VAL-FLOW-15", len(joined) == len(T["order_items"]),
       f"the join gives {len(joined):,} rows, the same as order_items; that is "
       f"{len(joined)/len(T['orders']):.2f} times the {len(T['orders']):,} orders, which is what "
       f"an order total worked out after the join would be multiplied by")

# VAL-FLOW-09 to -12 compare the two sources row by row, so they need the combined
# frame from before deduplication. That frame only exists inside the master notebook.
# The detector itself is written and tested here, so at assembly it is wired up rather
# than invented.

def find_conflicts(combined, key):
    """Keys where the two sources give different non-missing values for the same field.

    `combined` is the concatenation of the two normalised sources, before
    deduplication, carrying a `source_system` column.
    """
    shared = combined[combined.duplicated(key, keep=False)]
    found = []
    for field in shared.columns.drop([key, "source_system"]):
        distinct = shared.groupby(key)[field].nunique(dropna=True)
        for k in distinct[distinct > 1].index:
            found.append({"key": k, "field": field})
    return pd.DataFrame(found, columns=["key", "field"])


def count_overlap(combined, key):
    """How many keys came from both files.

    Count distinct keys, not marked rows: within-source duplicates would make
    a row count read 513 orders instead of 500.
    """
    return combined.loc[combined.source_system == "both", key].nunique()


# VAL-FLOW-11 — the negative control. A detector that never fires and a data set
# with no conflicts produce the same output, so the detector is shown working first.
fixture = pd.DataFrame({
    "order_id":      ["A1", "A1", "B2", "B2"],
    "order_total":   ["100.00", "100.00", "250.00", "999.00"],   # B2 disagrees
    "currency":      ["AUD", "AUD", "AUD", "AUD"],
    "source_system": ["JSON", "XML", "JSON", "XML"],
})
planted = find_conflicts(fixture, "order_id")
repaired = fixture.copy()
repaired.loc[3, "order_total"] = "250.00"
clean = find_conflicts(repaired, "order_id")

record("VAL-FLOW-11", len(planted) == 1 and len(clean) == 0,
       f"planted conflict detected: {planted.to_dict('records')}; "
       f"same frame with the conflict repaired: {len(clean)} conflicts")

# VAL-FLOW-09, -10 and -12 need WP2's pre-deduplication frames, which exist only
# inside the master notebook. They are recorded as NOT RUN so they appear in the
# exported register with their reason, rather than vanishing from it.
for vid, why, plan in [
    ("VAL-FLOW-09",
     "needs the combined frame from before deduplication",
     "run find_conflicts(combine_sources(t), key) on all four shared tables at assembly"),
    ("VAL-FLOW-10",
     "needs both copies of each within-source duplicate, which dedup removes",
     "compare the duplicate pairs on the combined frame at assembly"),
    ("VAL-FLOW-12",
     "needs source_system on <table>_marked; the CSVs correctly do not carry it",
     "run count_overlap(<table>_marked, key) and compare against the recomputed intersection"),
]:
    record(vid, None, why, plan)


VAL-FLOW-01        PASS     orders has 5,000 rows; the two files together hold 5,000 different keys
VAL-FLOW-02        PASS     order_items has 15,685 rows; the two files together hold 15,685 different keys
VAL-FLOW-03        PASS     customers has 500 rows; the two files together hold 500 different keys
VAL-FLOW-04        PASS     deliveries has 5,000 rows; the two files together hold 5,000 different keys
VAL-FLOW-05        PASS     products has 1,000 rows; the two files together hold 1,000 different keys
VAL-FLOW-06        PASS     product_reviews has 7,000 rows; the two files together hold 7,000 different keys
VAL-FLOW-07        PASS     deduplication removes 68 rows from the JSON and 68 from the XML
VAL-FLOW-08        PASS     rows with no source: {'orders': 0, 'order_items': 0, 'customers': 0, 'deliveries': 0, 'products': 0, 'product_reviews': 0}
VAL-FLOW-13        PASS     columns that are entirely NaN: none
VAL-FLOW-14        PASS     items per order run 1 to 5; the biggest cart

**Observed result / status / interpretation.** Every row count equals the union of business
keys derived from the raw files in the same cell, and no exported row lacks a source. Items per
order run within a bound taken from the source rather than from a number we chose, so the check
cannot inherit the 1–10 duplication artefact.

Two checks here earn their place beyond the counting. **VAL-FLOW-13** has a real result rather
than a formal one: before WP4 landed, three columns were entirely sentinel, and WP2 reported a
fault of exactly the kind this catches — §4.1 called the extractor and then overwrote the column
with a leftover placeholder. That raises nothing and changes no row count; an all-sentinel column
is the only symptom. **VAL-FLOW-15** measures the join fan-out rather than only avoiding it: the
join returns the item row count, 3.14× the order count, which is the factor by which an
order-level total computed after the join would be inflated.

VAL-FLOW-09, -10 and -12 are **not run here**, and are not counted as passes. They compare the
two sources row by row and need the combined frame from before deduplication, which exists only
in the master. The detector is written and unit-tested above so assembly wires it up rather than
invents it.

**What would have made this section fail.** A row count that disagrees with the union, an
exported row with no source, a fourth all-sentinel column, or a join that returns more rows than
`order_items` holds.

### 6.4 Arithmetic checks (`VAL-ARITH-...`)

Work the money out again and see if it matches. All of these use `money_round()` and
`same_number()` from §0.2.

| ID | What it checks | Table | Passes when | If it fails |
|---|---|---|---|---|
| VAL-ARITH-01 | `line_revenue` = quantity × unit price | order_items | within 0.01 | Line money is wrong, and every total built on it is wrong too |
| VAL-ARITH-02 | `order_price` = the sum of that order's line revenues | orders, order_items | within 0.01 | The order total does not match the lines it is made of |
| VAL-ARITH-03 | `order_total` = order price − discount + delivery. Tax is reported on its own and never added | orders | within 0.01 | Reading the discount as dollars instead of a percentage only reproduces the rows where the discount is zero |
| VAL-ARITH-04 | `tax_amount` = order price ÷ 11, worked out before the discount | orders | within 0.01 | The price already includes GST, so dividing by 11 pulls the tax back out. Multiplying by 0.1 would charge it twice |
| VAL-ARITH-05 | `coupon_discount` only holds values seen in this package | orders | all known | A percentage has been read as an amount, or an unexpected value crept in |
| VAL-ARITH-06 | `delay_days` = delivered date − promised date, but never below zero | deliveries | exact match | Early deliveries record 0, not a negative number. The plain difference does not reproduce the column |
| VAL-ARITH-07 | `review_length_chars` and `review_word_count` are counted from `review_body_clean` | product_reviews | exact match | The counts were taken from the messy raw text instead of the cleaned text (DEC-019) |

In [13]:
# --- §6.4 checks ---
oi = T["order_items"]
qty   = pd.to_numeric(oi.quantity)
price = pd.to_numeric(oi.unit_price)
line  = pd.to_numeric(oi.line_revenue)

# VAL-ARITH-01 — line revenue is quantity times unit price.
diff = ~same_number(line, money_round(qty * price))
record("VAL-ARITH-01", not diff.any(),
       f"{int(diff.sum())} of {len(oi):,} rows are more than 0.01 out")

o = T["orders"]
order_price = pd.to_numeric(o.order_price)
discount    = pd.to_numeric(o.coupon_discount)
delivery    = pd.to_numeric(o.delivery_charges)
total       = pd.to_numeric(o.order_total)
tax         = pd.to_numeric(o.tax_amount)

# VAL-ARITH-02 — order price is the sum of that order's line revenues.
per_order = money_round(pd.DataFrame({"oid": oi.order_id, "line": money_round(qty * price)})
                        .groupby("oid").line.sum())
lookup = o.set_index("order_id").order_price.astype(float)
diff = ~same_number(lookup, per_order.reindex(lookup.index))
record("VAL-ARITH-02", not diff.any(),
       f"{int(diff.sum())} of {len(o):,} orders are more than 0.01 out")

# VAL-ARITH-03 — the discount is a percentage, and tax is never added on top.
diff = ~same_number(total, money_round(order_price * (1 - discount / 100) + delivery))
record("VAL-ARITH-03", not diff.any(),
       f"{int(diff.sum())} of {len(o):,} orders are more than 0.01 out")

# VAL-ARITH-04 — the price already includes GST, so dividing by 11 pulls the tax back out.
diff = ~same_number(tax, money_round(order_price / 11))
record("VAL-ARITH-04", not diff.any(),
       f"{int(diff.sum())} of {len(o):,} orders are more than 0.01 out")

# VAL-ARITH-05 — the allowed set comes from the raw files, not from a list we chose.
# The earlier version passed unconditionally, which meant it could never fail.
raw_discounts = {float(src["header"]["couponDiscount"]) for src in jdata["orders"]}
for header in xroot.iter("Header"):
    node = header.find("Coupon_Discount")
    if node is not None and node.text:
        raw_discounts.add(float(node.text.replace("%", "").strip()))

seen = set(discount.unique())
record("VAL-ARITH-05", seen <= raw_discounts,
       f"export holds {sorted(int(v) for v in seen)}; the raw files hold "
       f"{sorted(int(v) for v in raw_discounts)}; unexpected values: "
       f"{sorted(seen - raw_discounts) or 'none'}")

# VAL-ARITH-08 — sensible numeric ranges. The specification asks for these
# alongside the allowed-value checks, and the register had none.
rv = T["product_reviews"]
rating = pd.to_numeric(rv.rating)
votes = pd.to_numeric(rv.helpful_votes)
money_columns = {
    "orders.order_price": order_price, "orders.order_total": total,
    "orders.delivery_charges": delivery, "orders.tax_amount": tax,
    "order_items.unit_price": price, "order_items.line_revenue": line,
}
out_of_range = {
    "rating outside 1-5": int(((rating < 1) | (rating > 5)).sum()),
    "quantity below 1": int((qty < 1).sum()),
    "helpful_votes negative": int((votes < 0).sum()),
    "negative money": sum(int((v < 0).sum()) for v in money_columns.values()),
}
record("VAL-ARITH-08", not any(out_of_range.values()),
       f"rating {rating.min()}-{rating.max()}, quantity from {qty.min()}, "
       f"helpful_votes from {votes.min()}, no negative money; violations {out_of_range}")

# Negative control for VAL-ARITH-04. The tax check only means something if the
# wrong formula fails. Adding GST on top instead of dividing it out should match
# almost nothing.
wrong_tax = money_round(order_price * 1.1 + delivery)
n_wrong = int(same_number(total, wrong_tax).sum())
print(f"{'(control)':18s} INFO  adding GST on top instead of dividing it out reproduces "
      f"{n_wrong:,} of {len(o):,} order totals — the correct formula reproduces "
      f"{int(same_number(total, money_round(order_price * (1 - discount / 100) + delivery)).sum()):,}")

# VAL-ARITH-06 — an early delivery records 0, not a negative number.
dl = T["deliveries"].copy()
for col in ["promised_date", "delivered_date"]:
    dl[col] = pd.to_datetime(dl[col])
gap = (dl.delivered_date - dl.promised_date).dt.days.clip(lower=0)
match = pd.to_numeric(dl.delay_days) == gap
record("VAL-ARITH-06", match.all(),
       f"{int(match.sum()):,} of {len(dl):,} rows match max(0, delivered - promised)")

# VAL-ARITH-07 — the counts come from the cleaned review, not the raw one.
body = rv.review_body_clean
chars_ok = (pd.to_numeric(rv.review_length_chars) == body.str.len()).all()
words_ok = (pd.to_numeric(rv.review_word_count) == body.str.split().str.len()).all()
record("VAL-ARITH-07", chars_ok and words_ok,
       f"character counts match: {chars_ok}; word counts match: {words_ok}")


VAL-ARITH-01       PASS     0 of 15,685 rows are more than 0.01 out
VAL-ARITH-02       PASS     0 of 5,000 orders are more than 0.01 out
VAL-ARITH-03       PASS     0 of 5,000 orders are more than 0.01 out
VAL-ARITH-04       PASS     0 of 5,000 orders are more than 0.01 out
VAL-ARITH-05       PASS     export holds [0, 5, 10, 15, 20, 25]; the raw files hold [0, 5, 10, 15, 20, 25]; unexpected values: none
VAL-ARITH-08       PASS     rating 1-5, quantity from 1, helpful_votes from 0, no negative money; violations {'rating outside 1-5': 0, 'quantity below 1': 0, 'helpful_votes negative': 0, 'negative money': 0}
(control)          INFO  adding GST on top instead of dividing it out reproduces 0 of 5,000 order totals — the correct formula reproduces 5,000
VAL-ARITH-06       PASS     5,000 of 5,000 rows match max(0, delivered - promised)
VAL-ARITH-07       PASS     character counts match: True; word counts match: True


**Observed result / status / interpretation.** Every monetary chain reconciles with no row
outside the published 0.01 tolerance, and `delay_days` matches `max(0, delivered − promised)` on
every row — the plain difference does not reproduce the column, because an early arrival records
zero rather than a negative.

These pass because they take two conventions from WP2 §4.0.3 rather than the obvious
implementations. Rounding uses Python's `round()`, not pandas `.round(2)`; comparison uses
`np.isclose(rtol=0, atol=0.01)`, not a hand-written `>`. Written the obvious way, the same data
reports 40 failures on `order_total`, none of which is a defect.

**The negative control is what makes VAL-ARITH-04 mean anything.** Adding GST on top instead of
dividing it out reproduces essentially none of the 5,000 totals, while the correct formula
reproduces all of them. Without that contrast, a passing tax check would be indistinguishable
from a check that cannot fail.

**What would have made this section fail.** A discount read as dollars — which would still
reproduce the zero-discount rows and so hide in plain sight — GST added rather than divided out,
a coupon value absent from the sources, a rating outside 1–5, or negative money.

### 6.5 Temporal checks (`VAL-TIME-...`)

Do the dates make sense in the order things actually happen?

| ID | What it checks | Table | Passes when | If it fails |
|---|---|---|---|---|
| VAL-TIME-01 | Order time ≤ dispatch date ≤ delivered date | orders, deliveries | no breaks | An impossible order of events, which usually means a date was read with the day and month swapped |
| VAL-TIME-02 | A review is not written before its order exists | product_reviews, orders | no breaks | The review points at an order that had not happened yet |
| VAL-TIME-03 | Every date reads cleanly under its own file's format — day-first for XML, ISO for JSON — with no dates turning into empty values | all dated tables | no empty dates | A wrong `dayfirst` setting turns dates into blanks quietly, and every date comparison after that is wrong |
| VAL-TIME-04 | Every `order_timestamp` falls inside 2018 | orders | all inside | A row from outside the period got in, or a date was read wrongly |

**One check we deliberately do not write.** We do **not** check that delivery happened before the
promised date. Some deliveries are late, and those rows have a `delay_days` and a `delay_reason`
filled in (`warehouse_congestion`, `carrier_capacity`, `weather`). Late delivery is how the
business went, not a data problem, so checking it would mark normal orders as failures. The
late rate belongs in the EDA instead.

For the same reason VAL-TIME-04 only looks at `order_timestamp`. Delivery dates run into January
2019 for orders placed at the very end of the year, which is fine.

In [14]:
# --- §6.5 checks ---
dl = T["deliveries"].copy()
for col in ["dispatch_date", "promised_date", "delivered_date"]:
    dl[col] = pd.to_datetime(dl[col])

order_time = pd.Series(pd.to_datetime(T["orders"].order_timestamp).values,
                       index=T["orders"].order_id.values)
dl["ordered"] = dl.order_id.map(order_time)

# VAL-TIME-01 — order, then dispatch, then delivery. This one is always true.
late_dispatch = int((dl.ordered.dt.normalize() > dl.dispatch_date).sum())
late_delivery = int((dl.dispatch_date > dl.delivered_date).sum())
record("VAL-TIME-01", late_dispatch + late_delivery == 0,
       f"dispatched before ordered: {late_dispatch}; delivered before dispatched: {late_delivery}")

# VAL-TIME-02 — a review cannot exist before its order.
rv = T["product_reviews"]
written = pd.to_datetime(rv.review_timestamp)
ordered = rv.order_id.map(order_time)
early = int((written < ordered).sum())
record("VAL-TIME-02", early == 0, f"{early} reviews are dated before their order")

# VAL-TIME-03 — a wrong dayfirst setting turns dates into blanks quietly.
blanks = sum(int(dl[c].isna().sum()) for c in ["dispatch_date", "promised_date", "delivered_date"])
blanks += int(written.isna().sum())
record("VAL-TIME-03", blanks == 0, f"{blanks} dates turned into blanks while being read")

# VAL-TIME-04 — only order_timestamp is held to 2018. Deliveries for orders placed
# at the end of December legitimately land in January 2019.
years = sorted(pd.to_datetime(T["orders"].order_timestamp).dt.year.unique())
record("VAL-TIME-04", years == [2018], f"order_timestamp years present: {years}")

# For context, not a check: how many deliveries were late.
print(f"{'(context)':18s} INFO  {int((dl.delivered_date > dl.promised_date).sum()):,} "
      f"deliveries arrived after the promised date — a business outcome, not a data problem")


VAL-TIME-01        PASS     dispatched before ordered: 0; delivered before dispatched: 0
VAL-TIME-02        PASS     0 reviews are dated before their order
VAL-TIME-03        PASS     0 dates turned into blanks while being read
VAL-TIME-04        PASS     order_timestamp years present: [np.int32(2018)]
(context)          INFO  528 deliveries arrived after the promised date — a business outcome, not a data problem


**Observed result / status / interpretation.** The chain `order ≤ dispatch ≤ delivered` holds
with no exceptions, no review predates its order, and no date turned into a blank while being
read — which is the evidence that both `dayfirst` conventions were applied to the right file.

**One check deliberately absent.** We do not assert that delivery beat the promised date. 528
deliveries were late, and those rows carry a populated `delay_days` and `delay_reason`
(`warehouse_congestion`, `carrier_capacity`, `weather`). That is a business outcome, not a data
defect; asserting it would report normal operations as failures, and the rate belongs in the EDA
instead. For the same reason VAL-TIME-04 constrains `order_timestamp` alone — delivery dates run
into January 2019 for orders placed at the very end of the period, which is correct.

**What would have made this section fail.** A delivery before its dispatch, a review before its
order, a silent `NaT` from the wrong `dayfirst`, or an order stamped outside 2018.

### 6.6 Text and multilingual checks (`VAL-TEXT-...`)

Was the text cleaned properly, and does the multilingual handling work? These check WP4's output.

| ID | What it checks | Table | Passes when | If it fails |
|---|---|---|---|---|
| VAL-TEXT-01 | The three narrative `_clean` fields have no HTML tags, markers, URLs or emoji left, and are all lower case | orders, products, product_reviews | none left | Cleaning failed and the mess reached the file |
| VAL-TEXT-01b | `delivery_note_clean` keeps the source's own capitals — it is a fixed category, not free text | deliveries | source spelling kept | Task 2 says not to lower-case a fixed category unless a rule says so, and no rule names this field. The `_clean` in the name is the source system's naming, not an instruction (DEC-018) |
| VAL-TEXT-02 | Bracketed wording that is not on the published marker list survives cleaning | orders, product_reviews | kept | A rule that removes every bracket also removes real customer wording (test case TXT-18) |
| VAL-TEXT-03 | `extracted_order_reference` is `[HC]ORD` plus exactly six digits, or the text `NaN` | product_reviews | all match | The pattern matches too much, and a wrong reference goes out looking valid |
| VAL-TEXT-04 | `extracted_product_sku` is `SKU-` plus letters or digits, or `NaN`. A broken SKU is rejected, not cut down to a valid start | product_reviews | all match | A broken SKU is exported as a good one (test case TXT-12) |
| VAL-TEXT-05 | `promo_code` is `B[1-5]SAVE-` plus two digits, or `NaN` | orders | all match | An invalid code goes out looking valid |
| VAL-TEXT-06 | The codes were pulled out of the **raw** text, not the cleaned text | orders, product_reviews | codes present | Cleaning deletes the reference, SKU and promo strings, so pulling them out afterwards returns `NaN` on every row (DEC-014) |
| VAL-TEXT-07 | `review_body_latin_analysis` keeps Latin letters including accents, turns non-Latin **letters** into spaces, leaves everything that is not a letter alone, and is `NaN` when no Latin letter is left. The check reports what survived **by character type**, so the rule is visible rather than assumed | product_reviews | no non-Latin letters; the sentinel fires whenever no Latin letter remains | Accents were dropped as if foreign, or the sentinel was not applied |
| VAL-TEXT-08 | `contains_non_latin_script` is `True` exactly when the cleaned review has a non-Latin letter | product_reviews | flag matches text | The flag and the text disagree. Note Polish, German and French use Latin letters with accents and must stay `False` — this is where a simple ASCII test goes wrong |
| VAL-TEXT-09 | `delay_reason` keeps `'none'` as a real value and no cleaning step turns it into `NaN` | deliveries | kept | A real value on thousands of rows is destroyed |
| VAL-TEXT-10 | Three fields hold one value only — `order_status`, `delivery_status`, `verified_purchase` — so §4's "completed" filter removes no rows, and this is written down as expected | orders, deliveries, product_reviews | noted | A filter that removes nothing is read as broken, or a real filter is quietly missing. None of the three can carry an EDA figure either |
| VAL-TEXT-12 | `coupon_code` is `B[1-5]SAVE-` plus two digits on every filled row | orders | all match | The shape VAL-TEXT-13 relies on is not what we think it is |
| VAL-TEXT-13 | Where a row has both `coupon_code` and `promo_code`, they agree — and no row has one without the other | orders | agree, none one-sided | See the note below |
| VAL-TEXT-14 | `extracted_order_reference` equals that row's own `order_id` | product_reviews | agree | The pattern matched a different order's reference inside the review |
| VAL-TEXT-15 | `extracted_product_sku` equals `products.product_sku` for that row's `product_id` | product_reviews, products | agree | Same idea as VAL-TEXT-14, checked through the foreign key |

**Why VAL-TEXT-13, -14 and -15 matter more than the rest.** Every other text check tests a
pattern against its own output — if the pattern is wrong, the check is wrong in the same way and
still passes. These three compare a value pulled out of free text against a value that got into
the table by a completely different route: a structured source column, the review's own
`order_id`, and the product catalogue. They are the only checks here that can tell a
right-looking wrong answer from a right one.

**What "Latin-script" means here, settled against the implementation.**
`Group001_text_functions.py` keeps a character when it is **not a letter**, or when its Unicode
name contains `LATIN`. So non-Latin *letters* become spaces, while digits, spaces and punctuation
stay — including CJK punctuation such as `。` and `，`, and the combining vowel marks used in
Devanagari and Arabic. That is the second of the two readings the spec allows: *remove non-Latin
letters*, not *keep only Latin letters*.

I checked the case this raises. If punctuation survives, could a review that is entirely
non-Latin end up as punctuation instead of the sentinel? No — the sentinel is decided by
"is there a Latin letter left", not by "is the text empty":

```
build_latin_analysis("包装很好")    ->  "NaN"
build_latin_analysis("包装很好。")  ->  "NaN"      punctuation left, still NaN
build_latin_analysis("。，、")      ->  "NaN"
```

All 18 published test cases pass, TXT-07 and TXT-09 among them. VAL-TEXT-07 therefore asserts
zero non-Latin *letters* and reports the other character types as evidence of the rule, rather
than treating them as a fault.


In [15]:
# --- §6.6 checks ---
import re
import unicodedata

NOISE = re.compile(r"<[^>]+>|https?://|\[(?:SYSTEM|CATALOGUE|VERIFIED_PURCHASE|SOURCE:|RATING:)"
                   r"|@store_support|&[a-z]+;|[\U0001F300-\U0001FAFF]")

# VAL-TEXT-01 — the three narrative fields must be clean and lower case.
narrative = [("orders", "customer_note_clean"),
             ("products", "product_description_clean"),
             ("product_reviews", "review_body_clean")]
noise_left = {f"{n}.{c}": int(T[n][c].str.contains(NOISE).sum()) for n, c in narrative}
case_left  = {f"{n}.{c}": int((T[n][c] != T[n][c].str.lower()).sum()) for n, c in narrative}
record("VAL-TEXT-01", not any(noise_left.values()) and not any(case_left.values()),
       f"noise left {noise_left}; rows not lower case {case_left}")

# VAL-TEXT-01b — delivery_note_clean is a fixed category, so it keeps its own capitals.
note = T["deliveries"].delivery_note_clean
record("VAL-TEXT-01b", int(note.str.contains(NOISE).sum()) == 0,
       f"{note.nunique()} different values, {int(note.str.contains(NOISE).sum())} with noise; "
       f"values: {sorted(note.unique())}")

# VAL-TEXT-03 / -04 / -05 / -12 — shape checks. NaN is a valid answer for all of them.
rv = T["product_reviews"]
SHAPES = [("VAL-TEXT-03", rv.extracted_order_reference, r"[HC]ORD\d{6}|NaN", "extracted_order_reference"),
          ("VAL-TEXT-04", rv.extracted_product_sku,     r"SKU-[A-Za-z0-9]+|NaN", "extracted_product_sku"),
          ("VAL-TEXT-05", T["orders"].promo_code,       r"B[1-5]SAVE-\d{2}|NaN", "promo_code"),
          ("VAL-TEXT-12", T["orders"].coupon_code,      r"B[1-5]SAVE-\d{2}|NaN", "coupon_code")]
for vid, col, pattern, label in SHAPES:
    bad = int((~col.str.fullmatch(pattern)).sum())
    record(vid, bad == 0,
           f"{label}: {bad} wrong shape; {int((col != 'NaN').sum()):,} of {len(col):,} filled in")

# VAL-TEXT-06 — the codes must be pulled from the raw text, before cleaning removes them.
still_there = int(rv.review_body_clean.str.contains(r"[HC]ORD\d{6}|SKU-[A-Za-z0-9]+").sum())
found = int((rv.extracted_order_reference != "NaN").sum())
record("VAL-TEXT-06", still_there == 0 and found > 0,
       f"references left in the cleaned text: {still_there}; references found: {found:,} "
       f"— so extraction ran before cleaning")


def has_non_latin_letter(text):
    """True if any letter is outside the Latin alphabet. Accents stay Latin."""
    for ch in text:
        if ch.isalpha():
            try:
                if not unicodedata.name(ch).startswith("LATIN"):
                    return True
            except ValueError:
                return True
    return False


# VAL-TEXT-07 — the rule is "remove non-Latin letters", not "keep only Latin letters".
# Digits, spaces and punctuation are left alone by design, so the check asserts that
# no non-Latin LETTER survived, and reports the other character types as evidence.
latin = rv.review_body_latin_analysis
leaked = int(latin[latin != "NaN"].map(has_non_latin_letter).sum())

kinds = {"accented Latin letters": 0, "non-Latin letters": 0,
         "punctuation and marks (kept by design)": 0}
for ch in {c for s in latin for c in s if ord(c) > 127}:
    if not ch.isalpha():
        kinds["punctuation and marks (kept by design)"] += 1
    elif has_non_latin_letter(ch):
        kinds["non-Latin letters"] += 1
    else:
        kinds["accented Latin letters"] += 1   # é, ü, ą — these are meant to stay

# The sentinel is decided by "is a Latin letter left", not by "is the text empty",
# so a review with only non-Latin text still returns NaN even when punctuation survives.
no_latin_left = ~rv.review_body_clean.map(
    lambda s: any(ch.isalpha() and not has_non_latin_letter(ch) for ch in s))
sentinel_ok = (latin[no_latin_left] == "NaN").all() if no_latin_left.any() else True

record("VAL-TEXT-07", leaked == 0 and sentinel_ok,
       f"{leaked} rows keep a non-Latin letter; sentinel correct on the "
       f"{int(no_latin_left.sum())} rows with no Latin letter; character types present: {kinds}")

# VAL-TEXT-08 — the flag must match the text. Polish, German and French use Latin
# letters with accents and must stay False; this is where an ASCII test goes wrong.
actual = rv.review_body_clean.map(has_non_latin_letter)
flag = rv.contains_non_latin_script.map({"True": True, "False": False})
wrong = int((flag != actual).sum())
accented = rv[rv.language_code.isin(["pl", "de", "fr", "it", "es", "nl", "pt"])]
false_alarms = int(accented.contains_non_latin_script.map({"True": True, "False": False}).sum())
record("VAL-TEXT-08", wrong == 0 and false_alarms == 0,
       f"{wrong} rows disagree; flag is True on {int(flag.sum())}, text says {int(actual.sum())}; "
       f"{false_alarms} of {len(accented):,} accented-Latin reviews wrongly flagged")

# VAL-TEXT-09 — 'none' is a real category, not a missing value.
reason = T["deliveries"].delay_reason
record("VAL-TEXT-09", (reason == "none").any(),
       f"delay_reason is 'none' on {int((reason=='none').sum()):,} rows and was not turned into NaN")

# VAL-TEXT-10 — three fields hold one value only, so the completed filter removes nothing.
single = {"orders.order_status": sorted(T["orders"].order_status.unique()),
          "deliveries.delivery_status": sorted(T["deliveries"].delivery_status.unique()),
          "product_reviews.verified_purchase": sorted(rv.verified_purchase.unique())}
record("VAL-TEXT-10", all(len(v) == 1 for v in single.values()),
       f"{single} — the completed filter removes no rows, which is expected here")

# VAL-TEXT-13 / -14 / -15 — the three cross-checks. Each compares a value pulled out
# of free text against a value that reached the table by a different route.
cc, pc = T["orders"].coupon_code, T["orders"].promo_code
both = (cc != "NaN") & (pc != "NaN")
one_sided = int(len(cc) - both.sum() - ((cc == "NaN") & (pc == "NaN")).sum())
disagree = int((cc[both] != pc[both]).sum())
record("VAL-TEXT-13", disagree == 0 and one_sided == 0,
       f"{int(both.sum()):,} rows have both and {disagree} disagree; {one_sided} rows have one but not the other")

mismatch = int((rv.extracted_order_reference != rv.order_id).sum())
record("VAL-TEXT-14", mismatch == 0,
       f"{mismatch} of {len(rv):,} reviews have a reference that is not their own order_id")

sku_of = T["products"].set_index("product_id").product_sku
mismatch = int((rv.extracted_product_sku != rv.product_id.map(sku_of)).sum())
record("VAL-TEXT-15", mismatch == 0,
       f"{mismatch} of {len(rv):,} reviews have a SKU that does not match their product")


VAL-TEXT-01        PASS     noise left {'orders.customer_note_clean': 0, 'products.product_description_clean': 0, 'product_reviews.review_body_clean': 0}; rows not lower case {'orders.customer_note_clean': 0, 'products.product_description_clean': 0, 'product_reviews.review_body_clean': 0}
VAL-TEXT-01b       PASS     2 different values, 0 with noise; values: ['Carrier scan reconciled', 'Delivered within promise']
VAL-TEXT-03        PASS     extracted_order_reference: 0 wrong shape; 7,000 of 7,000 filled in
VAL-TEXT-04        PASS     extracted_product_sku: 0 wrong shape; 7,000 of 7,000 filled in
VAL-TEXT-05        PASS     promo_code: 0 wrong shape; 1,873 of 5,000 filled in
VAL-TEXT-12        PASS     coupon_code: 0 wrong shape; 1,873 of 5,000 filled in
VAL-TEXT-06        PASS     references left in the cleaned text: 0; references found: 7,000 — so extraction ran before cleaning
VAL-TEXT-07        PASS     0 rows keep a non-Latin letter; sentinel correct on the 0 rows with no Latin lett

**Observed result / status / interpretation.** The three narrative fields are free of markup,
markers, URLs and emoji and are fully lower-cased. `delivery_note_clean` is asserted separately
as a structured category and now carries the source's own casing, which is the evidence that the
DEC-018 revert reached the files rather than living only in the register.

`contains_non_latin_script` is `True` on exactly the rows whose cleaned text carries a non-Latin
letter, and every review in a Latin-script language with diacritics correctly stays `False` —
that is the discrimination a naive ASCII test fails. `build_latin_analysis` keeps punctuation and
combining marks by design: the published rule removes non-Latin *letters*, and the sentinel is
decided by whether a Latin letter remains, not by whether the text is empty, so a wholly
non-Latin review still returns `NaN` even when punctuation survives.

**VAL-TEXT-13, -14 and -15 are the strongest checks in the register.** Every other text check
compares an extraction to the pattern that produced it, so a wrong pattern fails the same way
twice and still passes. These three compare a value pulled from free text against a value that
reached the table by an entirely different route — a structured source column, the review's own
`order_id`, and the product catalogue reached through the foreign key. They are the only checks
here that can distinguish a right-looking wrong answer from a right one, and all three come out
with no disagreements and no one-sided rows.

**What would have made this section fail.** Markup surviving cleaning, an extraction run on the
cleaned text instead of the raw text, diacritics stripped as if foreign, `delay_reason == 'none'`
mapped to the sentinel, or an extracted reference that does not match its own row.

### 6.7 Literal `NaN` reminder

Where the spec says a missing piece of text should be written as `NaN`, it means the three
letters `N`, `a`, `N` — not an empty cell, not Python's `None`, not a float NaN.

Two traps:

- Reading the file back with the normal settings turns that text into an empty value, and then
  the check cannot see it. Always read with `keep_default_na=False`.
- Turning an empty value into text does **not** produce `NaN` — pandas 2 gives lowercase `nan`.
  The text has to be filled in before anything is turned into a string, never produced by the
  conversion.

Fields that can hold the sentinel: `orders.coupon_code`, `orders.promo_code`,
`orders.customer_note_clean`, `deliveries.delivery_note_clean`,
`products.product_description_clean`, `product_reviews.review_body_clean`,
`product_reviews.review_body_latin_analysis`, `product_reviews.extracted_order_reference`,
`product_reviews.extracted_product_sku`.

| ID | What it checks | Table | Passes when | If it fails |
|---|---|---|---|---|
| VAL-TEXT-11 | In every field above, a missing value is the three letters `NaN`, checked on the exported file read with `keep_default_na=False` | orders, deliveries, products, product_reviews | all match | The required sentinel is not in the submitted files |

In [16]:
# --- §6.7 checks ---
# The tables were read with keep_default_na=False in §0.1, so the text NaN is
# still visible as three characters rather than an empty value.

SENTINEL_FIELDS = [("orders", "coupon_code"), ("orders", "promo_code"),
                   ("orders", "customer_note_clean"),
                   ("deliveries", "delivery_note_clean"),
                   ("products", "product_description_clean"),
                   ("product_reviews", "review_body_clean"),
                   ("product_reviews", "review_body_latin_analysis"),
                   ("product_reviews", "extracted_order_reference"),
                   ("product_reviews", "extracted_product_sku")]

counts, mixed = {}, []
for name, col in SENTINEL_FIELDS:
    series = T[name][col]
    counts[f"{name}.{col}"] = int((series == "NaN").sum())
    if (series == "").any():
        mixed.append(f"{name}.{col}")   # an empty cell where the sentinel was expected

record("VAL-TEXT-11", not mixed,
       f"NaN counts {counts}; fields mixing an empty cell with the sentinel: {mixed or 'none'}")


VAL-TEXT-11        PASS     NaN counts {'orders.coupon_code': 3127, 'orders.promo_code': 3127, 'orders.customer_note_clean': 0, 'deliveries.delivery_note_clean': 0, 'products.product_description_clean': 0, 'product_reviews.review_body_clean': 0, 'product_reviews.review_body_latin_analysis': 0, 'product_reviews.extracted_order_reference': 0, 'product_reviews.extracted_product_sku': 0}; fields mixing an empty cell with the sentinel: none


**Observed result / status / interpretation.** Read back with `keep_default_na=False`, the
sentinel appears as three characters wherever expected and no field mixes an empty cell with it.
Reading the files the ordinary way turns that text into a null and the check becomes blind, which
is why the read options in §0.1 are part of the check rather than a convenience.

**What would have made this section fail.** An empty cell where the sentinel belongs, or the
lowercase `nan` that pandas produces when a null is cast to text instead of the sentinel being
filled in first.

### How each result gets written down

One line per check, right after the code that runs it:

```
VAL-PK-01     PASS   5,000 different order_id, 0 blank, matches the number worked out from the sources
VAL-FLOW-09   PASS   3,259 shared keys, 59 columns, 0 conflicts after normalising
VAL-FLOW-11   PASS   planted conflict found in the test table; 0 in the real data
VAL-ARITH-03  FAIL   40 rows outside 0.01  —  what it means and what we propose doing
```

A failure gets its evidence and a suggested fix. The rubric gives marks for a real failed check
that finds the problem and says what to do about it, and no marks for making a number up so the
check passes. An all-green register is not the goal.

In [17]:
register = pd.DataFrame(RESULTS, columns=["id", "status", "evidence", "note"])

if register.empty:
    print("No checks have run yet.")
else:
    print(register.status.value_counts().to_dict())
    out_path = OUTPUT_DIR / f"{GROUP_ID}_validation_register.csv"
    register.to_csv(out_path, index=False)
    print(f"written -> {out_path}")

register

{'PASS': 65, 'NOT RUN': 3}
written -> outputs_wip_yandu/Group001_validation_register.csv


,id,status,evidence,note
0,VAL-SCHEMA-orders,PASS,"23 columns, order matches the dictionary",
1,VAL-SCHEMA-order_items,PASS,"6 columns, order matches the dictionary",
2,VAL-SCHEMA-customers,PASS,"20 columns, order matches the dictionary",
3,VAL-SCHEMA-deliveries,PASS,"20 columns, order matches the dictionary",
4,VAL-SCHEMA-products,PASS,"21 columns, order matches the dictionary",
...,...,...,...,...
63,VAL-TEXT-10,PASS,"{'orders.order_status': ['Completed'], 'delive...",
64,VAL-TEXT-13,PASS,"1,873 rows have both and 0 disagree; 0 rows ha...",
65,VAL-TEXT-14,PASS,"0 of 7,000 reviews have a reference that is no...",
66,VAL-TEXT-15,PASS,"0 of 7,000 reviews have a SKU that does not ma...",
